# C01 同次结果评价
先运行A，再运行B01，最后从普通新内核运行本Notebook。
本Notebook调用`03_eval.py`，默认读取`local_review/B01_candidate`，输出`local_review/C01_candidate`，保留仓库原冻结结果。
完整原始数据复刻入口是`python scripts/reproduce_abc.py`；环境依据GitHub原requirements与Python最低版本声明。


<!-- C01旧说明保留
# INT6163 第8组 Part C：实验设计与评价 + 数字守门

本 notebook 只编排最终评价。核心逻辑在 `03_eval.py`：Test 标签只读取一次，所有模型共用 `K=ceil(30%×8,238)=2,472`，并列分数按 `row_id` 升序决胜。

运行后统一生成 `outputs/metrics.json`、T3/T4/T5、F3/F4。`metrics.json` 是报告与 PPT 的唯一数字真值源。
-->

# INT6163 第8组 Part C：实验设计与评价及数字核对

本Notebook调用同目录03_eval.py。Test标签在每次最终评价中只加载一次；各模型共用K=ceil(30%×8,238)=2,472，同分按row_id升序。

默认同次B01输入位于local_review/B01_candidate；C输出的metrics.json、T3/T4/T5、F3/F4位于local_review/C01_candidate的outputs/figures下。所有表图读取这份JSON，原根目录冻结结果保留。


In [ ]:
# [C01 Notebook原代码格1开始]
# %run 03_eval.py
# [C01 Notebook原代码格1结束]
from pathlib import Path
import json
import runpy
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "03_eval.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
old_argv = sys.argv[:]
try:
    sys.argv = [str(ROOT / "03_eval.py")]
    c_result = runpy.run_path(str(ROOT / "03_eval.py"), run_name="__main__")
finally:
    sys.argv = old_argv
metrics_path = c_result["METRICS"]


## 数字守门检查

下面只读取锁定后的 `metrics.json`，用于核对主工作点、业务增益和 duration 消融。

In [ ]:
# [C01 Notebook原代码格2开始]
# import json
# from pathlib import Path
#
# metrics = json.loads(Path('outputs/metrics.json').read_text(encoding='utf-8'))
# print('Test rows:', metrics['protocol']['test_rows'])
# print('Test positives:', metrics['protocol']['test_positive_rows'])
# print('K at 30%:', metrics['protocol']['k_at_30pct'])
# print('Main model:', metrics['selected_model'])
# print('Main PR-AUC:', metrics['models']['logistic']['pr_auc'])
# print('Main hits@K:', metrics['models']['logistic']['hits_at_k'])
# print('Extra hits vs rule:', metrics['business']['extra_hits_vs_rule'])
# print('Invalid calls avoided vs rule:', metrics['business']['invalid_calls_avoided_vs_rule'])
# print('Missed subscriptions@K:', metrics['business']['missed_subscriptions_at_k'])
# print('Duration AP inflation:', metrics['ablation']['test_pr_auc_absolute_difference'])
# [C01 Notebook原代码格2结束]
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
print("当前同次结果：", metrics_path)
print(json.dumps({"protocol": metrics["protocol"], "business": metrics["business"]}, ensure_ascii=False, indent=2))
